In [ ]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
from arcpy import metadata as md

In [ ]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# List of geodatabases containing items to export metadata for (individual HTML file output for each individual item in gdb)
gdbs = [
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec_USCaribbean.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb"
]

# Output folder to export XML metadata files in
out_root = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\metadata\metadata_xml_files"

In [ ]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [ ]:
# ── EXPORT ITEM METADATA IN XML FORMAT ───────────────────────────────────────────────────────────────

# Loop through each geodatbase (gdb) in gdbs[] list (created in path block)
for gdb in gdbs:
    # set arcpy workspace to current gdb in loop iteration so arcpy.List⧆() functions read from it
    arcpy.env.workspace = gdb

    # extract name of gdb (used for failure messages)
    gdb_name = os.path.splitext(os.path.basename(gdb))[0]

    # collect all feature classes/rasters/tables/etc. in gdb
    items = []
    items += arcpy.ListFeatureClasses() or []
    items += arcpy.ListRasters() or []
    items += arcpy.ListTables() or []
    # also collect feature classes inside feature datasets (if applicable)
    for fds in arcpy.ListDatasets(feature_type="Feature") or []:
        for fc in arcpy.ListFeatureClasses(feature_dataset=fds) or []:
            items.append(os.path.join(fds, fc))

    # loop through each individual item in gdb
    for item in items:
        # build full path to item
        full_path = os.path.join(gdb, item)
        # build output xml file path from item name
        out_xml = os.path.join(out_root, os.path.basename(item) + ".xml")

        # wrap each export so one bad item does not stop entire batch
        try:
            # load item metadata into a Metadata object
            m = md.Metadata(full_path)
            # saveAsXML writes complete ArcGIS-format metadata as-is (all content, no schema translation, no content-sensitive filtering)
            m.saveAsXML(out_xml)
            print(f"✅ Exported: {out_xml} \n")
        # print warning message if export fails
        except Exception as e:
            print(f"❌[FAILURE] {item} in {gdb_name}.gdb: {e} \n")